In [ ]:
import csv
import time
import json
import boto3
import base64

from pathlib import Path
from PIL import Image
from trp import Document
from IPython.display import display

MODEL_ID = "us.anthropic.claude-sonnet-4-6"
MAX_TOKENS = 1000

textract_client = boto3.client('textract')
bedrock_client = boto3.client('bedrock-runtime')

In [ ]:
def _media_type_for(path: str) -> str:
    ext = Path(path).suffix.lower().lstrip(".")
    return {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png",
            "gif": "image/gif", "webp": "image/webp"}.get(ext, "image/jpeg")


def get_textract_text(image_path: str):
    with open(image_path, "rb") as image_file:
        document_bytes = image_file.read()

    response = textract_client.analyze_document(
        Document={"Bytes": document_bytes},
        FeatureTypes=["TABLES", "FORMS", "SIGNATURES", "LAYOUT"],
    )

    return response


def encode_image(label_image_path: str):
    with open(label_image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def claude_response(base64_img, prompt, media_type="image/jpeg"):
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": MAX_TOKENS,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image",
                     "source": {"type": "base64", "media_type": media_type, "data": base64_img}},
                    {"type": "text", "text": prompt},
                ],
            }
        ],
    }

    response = bedrock_client.invoke_model(
        modelId=MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(body),
    )

    return json.loads(response["body"].read())

In [ ]:
vaccination_card_doc = r'images/analyze doc/vaccination_card.png'
image = Image.open(vaccination_card_doc)

if image.mode == 'RGBA':
    image = image.convert('RGB')

display(image)

In [ ]:
start_time = time.time()
textract_output = get_textract_text(vaccination_card_doc)
textract_time = time.time() - start_time

print(f"\nTime taken for Textract: {textract_time:.2f} seconds")

text = ''

for item in textract_output['Blocks']:
    if item['BlockType'] == 'LINE':
   
        text += item['Text']

print (text)

In [ ]:

base64_img = encode_image(vaccination_card_doc)
prompt = '''
What's the text in the image ? are there any tables in it the image ?
Analyze the document, what do you think the document might be
Give all the extracted text, the tables (in csv) and your analysis separately.

'''

start_time = time.time()
claud = claude_response(base64_img, prompt, _media_type_for(vaccination_card_doc))
claude_time = time.time() - start_time

print('\nClaude  Response: \n')
print(claud['content'][0]['text'])
print(f"\nTime taken for Claude: {claude_time:.2f} seconds")

In [ ]:
doc = Document(textract_output)


def save_table_as_csv(table, filename):
    with open(filename, mode="w", newline="", encoding="utf-8") as csv_file:
        writer = csv.writer(csv_file)
        for row in table.rows:
            row_data = [cell.text.strip() for cell in row.cells]
            writer.writerow(row_data)
    print(f"Table saved as {filename}")


for page_num, page in enumerate(doc.pages, start=1):
    for table_num, table in enumerate(page.tables, start=1):
        filename = f"table_page_{page_num}_table_{table_num}.csv"
        save_table_as_csv(table, filename)

In [ ]:
receipt =  r'images/analyze expense/default_receipt_1.jpeg'

image = Image.open(receipt)
display (image)

In [ ]:

def analyze_expense (image_path : str):
    with open (image_path , "rb") as image_file:
        document_bytes = image_file.read()

    response = textract_client.analyze_expense(
        Document = {'Bytes' : document_bytes}

    )
        
    return response

In [ ]:
start_time = time.time()
textract_expense = analyze_expense (receipt)
textract_time = time.time() - start_time


print(f'Time taken by Textract {textract_time: .2f} seconds')


# print (textract_expense)

In [ ]:
invoice_summary = ''
expense_details = ''

for x in textract_expense['ExpenseDocuments'][0]:
    if x == 'SummaryFields':
        for summary in textract_expense['ExpenseDocuments'][0]['SummaryFields']:
            invoice_summary += (summary['Type']['Text'] + ': \n' + summary['ValueDetection']['Text'] + '\n')
    if x == 'LineItemGroups':
        for line in textract_expense['ExpenseDocuments'][0]['LineItemGroups']:
           for item in line["LineItems"]:
               for l in item['LineItemExpenseFields']:
                   expense_details += (l['Type']['Text'] + ':' + l['ValueDetection']['Text'] + '\n')

In [ ]:
print (invoice_summary)

In [ ]:
print (expense_details)

In [ ]:
base64_img = encode_image(receipt)
prompt = '''
Extract all information from this receipt/invoice and analyze the expense.
Give me the extracted information and the analysis of the expense separately.
'''

start_time = time.time()
claud = claude_response(base64_img, prompt, _media_type_for(receipt))
claude_time = time.time() - start_time

print('\nClaude  Response: \n')
print(claud['content'][0]['text'])
print(f"\nTime taken for Claude: {claude_time:.2f} seconds")

In [ ]:

def analyze_id (image_path : str):
    with open (image_path , "rb") as image_file:
        document_bytes = image_file.read()

    response = textract_client.analyze_id(
        DocumentPages = [{'Bytes' : document_bytes}]

    )
        
    return response

In [ ]:
id_doc =  r'images/analyze id/DLRegular.png'
image = Image.open(id_doc)

if image.mode == 'RGBA':
    image = image.convert('RGB')
display (image)

In [ ]:
start_time = time.time()
id_detials = analyze_id (id_doc)
textract_time = time.time() - start_time

print(f'Time taken by Textract {textract_time: .2f} seconds')


# print (id_detials)

In [ ]:
id_details = ''
id_text = ''

for x in id_detials:
    if x == 'IdentityDocuments':
        for y in id_detials['IdentityDocuments'][0]['IdentityDocumentFields']:
            id_details += y['Type']['Text'] + ':' + y['ValueDetection']['Text'] + '\n'
        
        for z in id_detials['IdentityDocuments'][0]['Blocks']:
            if z['BlockType'] == 'LINE':
                
                id_text += z['Text'] + '\n' 

In [ ]:
print(id_details)

In [ ]:
print (id_text)

In [ ]:

base64_img = encode_image(id_doc)
prompt = '''
Extract all the information from this document and tell me 
what type of identity document is this ?
Provide the extracted information and your analysis separately

'''

start_time = time.time()
claud = claude_response(base64_img, prompt, _media_type_for(id_doc))
claude_time = time.time() - start_time

print('\nClaude  Response: \n')
print(claud['content'][0]['text'])
print(f"\nTime taken for Claude: {claude_time:.2f} seconds")

# Enhancing Claude's Capabilities with Textract

In [ ]:
vaccination_card_doc = r'images/analyze doc/vaccination_card.png'
image = Image.open(vaccination_card_doc)

if image.mode == 'RGBA':
    image = image.convert('RGB')

display(image)

In [ ]:
print (f'Raw Text: {text}')

base64_img = encode_image(vaccination_card_doc)
prompt = f'''
This is the text extracted from the image using Textract: {text}
From your analysis, what's the text in the image ? are there any tables in it the image ?
What do you think the document might be.
Use the provided text to help yourself in information extraction and use that as context
for your analysis.
Give all the extracted text, the tables (in the form of csv) and your analysis separately.
'''

start_time = time.time()
claud = claude_response(base64_img, prompt, _media_type_for(vaccination_card_doc))
claude_time = time.time() - start_time

print('\nClaude  Response: \n')
print(claud['content'][0]['text'])
print(f"\nTime taken for Claude: {claude_time:.2f} seconds")

In [ ]:
receipt =  r'images/analyze expense/default_receipt_1.jpeg'
image = Image.open(receipt)
display (image)

In [ ]:

base64_img = encode_image(receipt)
prompt = f'''
This is the summary of the invoice/receipt: {invoice_summary}
and here are the details: {expense_details}
Extract all information from this receipt/invoice and analyze the expense.
Use the provided text to help yourself in information extraction and use
that as context for your analysis.
Give me the extracted information and the analysis of the expense separately.

'''

start_time = time.time()
claud = claude_response(base64_img, prompt, _media_type_for(receipt))
claude_time = time.time() - start_time

print('\nClaude  Response: \n')
print(claud['content'][0]['text'])
print(f"\nTime taken for Claude: {claude_time:.2f} seconds")

In [ ]:
id_doc =  r'images/analyze id/DLRegular.png'
image = Image.open(id_doc)

if image.mode == 'RGBA':
    image = image.convert('RGB')
display (image)

In [ ]:

base64_img = encode_image(id_doc)
prompt = f'''
Here are the ID details extracted from Textract:{id_details}
and here is all the text: {id_text}
Analyze the document, extract all the information and tell me 
what type of identity document this is ?
Use the provided text to help yourself in information extraction 
and use that as context for your analysis.
Provide the extracted information and your analysis separately

'''

start_time = time.time()
claud = claude_response(base64_img, prompt, _media_type_for(id_doc))
claude_time = time.time() - start_time

print('\nClaude  Response: \n')
print(claud['content'][0]['text'])
print(f"\nTime taken for Claude: {claude_time:.2f} seconds")